In [1]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score
import numpy as np

In [2]:
X,y = load_diabetes(return_X_y=True)

In [3]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=4)

<h1>Using SGD</h1>

In [8]:
from sklearn.linear_model import SGDRegressor
reg = SGDRegressor(penalty='l2',max_iter=500,eta0=0.1,learning_rate='constant',alpha=0.001)

In [9]:
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.40529028981656623
[ 5.31353020e+01 -1.39409755e+02  3.53266591e+02  2.60344076e+02
 -6.10585283e-02 -5.33351417e+01 -1.69040280e+02  1.37523197e+02
  3.23587732e+02  1.04331106e+02]
[173.00065059]


<h1>Using Ridge</h1>

In [10]:
from sklearn.linear_model import Ridge

reg = Ridge(alpha=0.001, max_iter=500,solver='sparse_cg')

In [11]:
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.46250101620275696
[  34.52192776 -290.84083874  482.40181671  368.06786928 -852.44872822
  501.5916069   180.11115477  270.76334438  759.73534797   37.49135792]
151.10198518246295


<h1>Optimizing Ridge Regression by GD</h1>

In [12]:
class MyRidgeGD:

    def __init__(self,epochs,lr,alpha):
        self.lr=lr
        self.epochs=epochs
        self.alpha=alpha
        self.coef_=None
        self.intercept_=None

    def fit(self,X_train,y_train):
        self.coef_=np.ones(X_train.shape[1])
        self.intercept_=0
        theta=np.insert(self.coef_,0,self.intercept_)

        X_train=np.insert(X_train,0,1,axis=1)

        for i in range(self.epochs):
            theta_der=np.dot(X_train.T,X_train).dot(theta)-np.dot(X_train.T,y_train)+self.alpha*theta
            theta=theta-self.lr*theta_der

        self.coef_=theta[1:]
        self.intercept_=theta[0]

    def predict(self,X_test):
        return np.dot(X_test,self.coef_)+self.intercept_

In [14]:
reg = MyRidgeGD(epochs=500,alpha=0.001,lr=0.005)

In [15]:
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.4738018280260915
[  46.65050914 -221.3750037   452.12080647  325.54248128  -29.09464178
  -96.47517735 -190.90017011  146.32900372  400.80267299   95.09048094]
150.8697531671347


## Ridge Regression being Optimized by Gradient Descent

Ridge Regression can be optimized using Gradient Descent by repeatedly updating the model parameters in the direction that reduces the Ridge loss.

The update rule is:

$$
\theta
\leftarrow
\theta
-
\eta
\left(
X^T X\theta
-
X^T y
+
\alpha\theta
\right)
$$

where:

- $\theta$ = model parameters
- $\eta$ = learning rate
- $X^T X\theta - X^T y$ = gradient of the squared-error term
- $\alpha\theta$ = gradient of the L2 regularization term
- $\alpha$ = regularization strength

The process is repeated for multiple iterations so that the parameters gradually move toward values that minimize the Ridge loss.

### Optimization Process

$$
\text{Initialize }\theta
\rightarrow
\text{Calculate gradient}
\rightarrow
\text{Update }\theta
\rightarrow
\text{Repeat}
$$

Thus, **Ridge Regression is optimized by Gradient Descent by repeatedly updating its coefficients to minimize prediction error while also penalizing large coefficients.**